# ESL Error Detector — Colab Training
Run each cell in order. Make sure your runtime is set to a GPU (Runtime → Change runtime type → T4 GPU).

In [1]:
# check GPU
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

GPU available: True
Device: Tesla T4


In [2]:
!pip install -q transformers datasets scikit-learn pandas

In [3]:
# mount drive to save checkpoint
from google.colab import drive
drive.mount('/content/drive')
SAVE_PATH = '/content/drive/MyDrive/esl_checkpoint'

Mounted at /content/drive


In [4]:
# load and preprocess dataset
from datasets import load_dataset
import pandas as pd

ds = load_dataset('512duncanl/wi_locness_detokenized', split='train')
df = pd.DataFrame({'original_text': ds['input'], 'corrected_text': ds['output']})
print(f'Loaded {len(df)} examples')
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/311 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/4.60M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/38692 [00:00<?, ? examples/s]

Loaded 38692 examples


,original_text,corrected_text
0,My town is a medium size city with eighty thou...,My town is a medium - sized city with eighty t...
1,It has a high density population because its s...,It has a high density population because of it...
2,"Despite of it is an industrial city, there are...","Although it is an industrial city, there are m..."
3,I recommend visiting the artificial lake in th...,I recommend visiting the artificial lake in th...
4,Pasteries are very common and most of them off...,Pasteries are very common and most of them off...


In [5]:
# build labeled classification dataset
# use original (erroneous) text as ERR, corrected text as OK
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
half = len(df) // 2

err_examples = pd.DataFrame({'text': df.iloc[:half]['original_text'], 'label': 1})
ok_examples  = pd.DataFrame({'text': df.iloc[half:]['corrected_text'], 'label': 0})

combined = pd.concat([err_examples, ok_examples], ignore_index=True)
combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset size: {len(combined)}')
print(combined['label'].value_counts())

Dataset size: 38692
label
1    19346
0    19346
Name: count, dtype: int64


In [6]:
# tokenize
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

ds = Dataset.from_pandas(combined[['text', 'label']])
split = ds.train_test_split(test_size=0.2)

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

tokenized = split.map(tokenize, batched=True)
print('Tokenization done')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/30953 [00:00<?, ? examples/s]

Map:   0%|          | 0/7739 [00:00<?, ? examples/s]

Tokenization done


In [7]:
# train
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=-1)
    return {
        'accuracy':  float((preds == labels).mean()),
        'f1':        f1_score(labels, preds),
        'precision': precision_score(labels, preds),
        'recall':    recall_score(labels, preds),
    }

args = TrainingArguments(
    output_dir=SAVE_PATH,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    compute_metrics=compute_metrics,
)

trainer.train()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.624747,0.613928,0.643623,0.627800,0.657619,0.600568
2,0.488258,0.651929,0.639618,0.551680,0.730835,0.443067
3,0.340690,0.818244,0.648663,0.605313,0.691313,0.538342


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2904, training_loss=0.49289500545043263, metrics={'train_runtime': 1153.1918, 'train_samples_per_second': 80.523, 'train_steps_per_second': 2.518, 'total_flos': 3075197542949376.0, 'train_loss': 0.49289500545043263, 'epoch': 3.0})

In [8]:
# evaluate and print final metrics
results = trainer.evaluate()
print('\n=== Final Eval Results ===')
for k, v in results.items():
    print(f'{k}: {v:.4f}')


=== Final Eval Results ===
eval_loss: 0.6138
eval_accuracy: 0.6436
eval_f1: 0.6274
eval_precision: 0.6580
eval_recall: 0.5995
eval_runtime: 31.4033
eval_samples_per_second: 246.4390
eval_steps_per_second: 7.7060
epoch: 3.0000


In [9]:
# save model and tokenizer to drive
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'Saved to {SAVE_PATH}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/esl_checkpoint


In [12]:
# quick sanity check
LABEL_LIST = ['OK', 'ERR']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

test_sentences = [
    ('She go to school yesterday.', 'ERR'),
    ('He enjoys playing basketball.', 'OK'),
    ('I has a apple.', 'ERR'),
    ('The cat sat on the mat.', 'OK'),
    ('Me and him went to the park.', 'ERR'),
    ('We are watching a movie tonight.', 'OK'),
    ('They was late to the meeting.', 'ERR'),
    ('This is a beautiful day.', 'OK'),
    ("Him don't like vegetables.", 'ERR'),
    ('She is going to the store.', 'OK'),
]

correct = 0
for text, expected in test_sentences:
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    pred = LABEL_LIST[torch.argmax(logits, dim=-1).item()]
    status = 'V' if pred == expected else 'X'
    print(f'{status} [{expected} -> {pred}] {text}')
    if pred == expected:
        correct += 1

print(f'\nAccuracy: {correct}/{len(test_sentences)}')

V [ERR -> ERR] She go to school yesterday.
V [OK -> OK] He enjoys playing basketball.
V [ERR -> ERR] I has a apple.
V [OK -> OK] The cat sat on the mat.
X [ERR -> OK] Me and him went to the park.
V [OK -> OK] We are watching a movie tonight.
V [ERR -> ERR] They was late to the meeting.
V [OK -> OK] This is a beautiful day.
V [ERR -> ERR] Him don't like vegetables.
V [OK -> OK] She is going to the store.

Accuracy: 9/10
